# DP2 — DiaObject Light Curves (DiaSource + ForcedPhotometry) — COSMOS DDF

**Author:** dagoret  
**Creation Date:** 2026-03-14  
**Last Date:** 2026-03-14  
**Version:** v1

## Purpose

Read the 3 CSV files generated by `2026-03-13_DP2_DDF_DiaObjects_Butler.ipynb`:
- `DiaObjects_COSMOS_nmin200.csv`
- `DiaSources_COSMOS_nmin200.csv`
- `ForcedSrc_COSMOS_nmin200.csv`

Then plot **multi-band light curves** (u, g, r, i, z, y) for a subset of DiaObjects  
selected by `nDiaSources > CUT_NDIASOURCES`.

Each figure shows one DiaObject with:
- **Marker `o`** : DiaSource detections (psf flux or AB magnitude)
- **Marker `+`** : Forced photometry
- **Error bars** for both
- **Independent y-axes** (twin axes) per band, offset vertically
- **Top x-axis** : Calendar date (YYYY-MM-DD)
- **Bottom x-axis** : MJD

## Input data
Generated by: `2026-03-13_DP2_DDF_DiaObjects_Butler.ipynb`  
Data folder: `data_DP2_DDF_DIAOBJ_BUTLER_01/`

---
## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

from astropy.time import Time

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update({
    "figure.figsize": (14, 8),
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 11,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
    "figure.titlesize": 18,
})

---
## 1. Parameters

In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = "data_DP2_DDF_DIAOBJ_BUTLER_01"

FILE_DIAOBJ   = f"{DATA_DIR}/DiaObjects_COSMOS_nmin200.csv"
FILE_DIASRC   = f"{DATA_DIR}/DiaSources_COSMOS_nmin200.csv"
FILE_FORCEDSRC = f"{DATA_DIR}/ForcedSrc_COSMOS_nmin200.csv"

# ─── Selection ────────────────────────────────────────────────────────────────
# Minimum number of DiaSource detections to select a DiaObject
CUT_NDIASOURCES = 300          # select objects with nDiaSources > this value

# Maximum number of light curves to display (to avoid overloading the notebook)
MAX_CURVES = 20                # set to None to plot all selected objects

# ─── Plot mode ────────────────────────────────────────────────────────────────
# 'flux'  → display psfFlux in nJy (or ADU)
# 'mag'   → display AB magnitude  (mag = -2.5*log10(flux/3631e9)  for flux in nJy)
PLOT_MODE = 'flux'             # 'flux' or 'mag'

# ─── Bands & cosmetics ────────────────────────────────────────────────────────
BANDS    = ['u', 'g', 'r', 'i', 'z', 'y']
COLORS   = {'u': '#7b2d8b', 'g': '#1f8b4c', 'r': '#e74c3c',
            'i': '#e67e22', 'z': '#2980b9', 'y': '#8e44ad'}
OFFSETS  = {'u': 5, 'g': 4, 'r': 3, 'i': 2, 'z': 1, 'y': 0}  # twin-axis spine offset (×60 px)

# ─── Flux → magnitude conversion ──────────────────────────────────────────────
# Rubin/LSST zero-point: flux in nJy  →  m_AB = -2.5 * log10(flux / 3631e9)
# equivalently          m_AB = 31.4 - 2.5 * log10(flux)  for flux in nJy
ZP_FLUX_NJY = 3631e9   # nJy corresponding to mag 0

def flux_to_mag(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive fluxes."""
    with np.errstate(divide='ignore', invalid='ignore'):
        mag = np.where(flux_nJy > 0,
                       -2.5 * np.log10(np.where(flux_nJy > 0, flux_nJy, np.nan) / ZP_FLUX_NJY),
                       np.nan)
    return mag

def flux_err_to_mag_err(flux_nJy, flux_err_nJy):
    """Propagate flux error to magnitude error (symmetric approximation)."""
    with np.errstate(divide='ignore', invalid='ignore'):
        mag_err = np.where(flux_nJy > 0,
                           (2.5 / np.log(10)) * np.abs(flux_err_nJy / np.where(flux_nJy > 0, flux_nJy, np.nan)),
                           np.nan)
    return mag_err

print(f"CUT_NDIASOURCES : {CUT_NDIASOURCES}")
print(f"MAX_CURVES      : {MAX_CURVES}")
print(f"PLOT_MODE       : {PLOT_MODE}")

---
## 2. Load data

In [ ]:
df_obj = pd.read_csv(FILE_DIAOBJ)
df_src = pd.read_csv(FILE_DIASRC)
df_frc = pd.read_csv(FILE_FORCEDSRC)

print(f"DiaObjects  : {len(df_obj):,} rows,  columns: {list(df_obj.columns[:6])} ...")
print(f"DiaSources  : {len(df_src):,} rows,  columns: {list(df_src.columns[:6])} ...")
print(f"ForcedSrc   : {len(df_frc):,} rows,  columns: {list(df_frc.columns[:6])} ...")

In [ ]:
# Quick sanity check on the nDiaSources distribution
print(df_obj['nDiaSources'].describe())
print(f"\nObjects with nDiaSources > {CUT_NDIASOURCES}: "
      f"{(df_obj['nDiaSources'] > CUT_NDIASOURCES).sum()}")

---
## 3. Select DiaObjects

In [ ]:
# Apply nDiaSources cut
mask_cut = df_obj['nDiaSources'] > CUT_NDIASOURCES
df_obj_sel = df_obj[mask_cut].copy()
df_obj_sel = df_obj_sel.sort_values('nDiaSources', ascending=False).reset_index(drop=True)

print(f"Selected DiaObjects (nDiaSources > {CUT_NDIASOURCES}): {len(df_obj_sel)}")

# Limit number of curves
if MAX_CURVES is not None:
    df_obj_plot = df_obj_sel.head(MAX_CURVES)
else:
    df_obj_plot = df_obj_sel

print(f"Will plot {len(df_obj_plot)} light curves  (MAX_CURVES={MAX_CURVES})")
display(df_obj_plot[['diaObjectId', 'ra', 'dec', 'nDiaSources']].head(20))

---
## 4. Helper: MJD ↔ Date conversion

In [ ]:
def mjd_to_datetime(mjd_array):
    """Convert an array of MJD (TAI or UTC, difference negligible for display) to
    numpy datetime64 objects suitable for matplotlib date axis."""
    t = Time(mjd_array, format='mjd', scale='tai')
    return t.to_datetime()   # returns list of datetime objects


def make_date_axis(ax_mjd, mjd_min, mjd_max, n_ticks=6):
    """Add a secondary x-axis on top showing calendar dates.
    
    Parameters
    ----------
    ax_mjd : matplotlib.axes.Axes
        The primary bottom axis with MJD values.
    mjd_min, mjd_max : float
        Range of the MJD axis.
    n_ticks : int
        Approximate number of date ticks.
    """
    ax_date = ax_mjd.twiny()          # new axis sharing same x-data space
    ax_date.set_xlim(ax_mjd.get_xlim())

    # Choose evenly spaced MJD tick positions
    tick_mjds = np.linspace(mjd_min, mjd_max, n_ticks)
    tick_labels = [Time(m, format='mjd', scale='tai').strftime('%Y-%m-%d')
                   for m in tick_mjds]

    ax_date.set_xticks(tick_mjds)
    ax_date.set_xticklabels(tick_labels, rotation=30, ha='left', fontsize=10)
    ax_date.xaxis.set_label_position('top')
    ax_date.set_xlabel('Date (UTC)', fontsize=11, labelpad=8)
    return ax_date

---
## 5. Helper: Plot one light curve

In [ ]:
def plot_lightcurve(diaobj_row, df_src_obj, df_frc_obj,
                    plot_mode='flux', bands=None, colors=None, offsets=None):
    """
    Plot a multi-band light curve for a single DiaObject.

    Parameters
    ----------
    diaobj_row : pd.Series
        Row from the DiaObjects DataFrame.
    df_src_obj : pd.DataFrame
        Subset of DiaSources for this diaObjectId.
    df_frc_obj : pd.DataFrame
        Subset of ForcedSrc for this diaObjectId.
    plot_mode : str
        'flux' or 'mag'.
    bands, colors, offsets : dicts
        Band list, color mapping, and y-axis offset index mapping.
    """
    if bands is None:
        bands = BANDS
    if colors is None:
        colors = COLORS
    if offsets is None:
        offsets = OFFSETS

    diaObjectId = diaobj_row['diaObjectId']
    ra  = diaobj_row['ra']
    dec = diaobj_row['dec']
    n   = int(diaobj_row['nDiaSources'])

    fig, ax_main = plt.subplots(figsize=(15, 7))
    ax_main.set_visible(True)

    # We will use ax_main only for the x-axis and frame;
    # each band gets its own twin y-axis.
    ax_main.set_xlabel('MJD (TAI)', fontsize=13)
    if plot_mode == 'flux':
        y_label_base = 'PSF Flux (nJy)'
    else:
        y_label_base = 'AB Magnitude'

    # Collect MJD range for date axis
    all_mjds = []
    if 'midpointMjdTai' in df_src_obj.columns:
        all_mjds.extend(df_src_obj['midpointMjdTai'].dropna().tolist())
    if 'mjd_visit' in df_frc_obj.columns:
        all_mjds.extend(df_frc_obj['mjd_visit'].dropna().tolist())

    if len(all_mjds) == 0:
        print(f"  WARNING: no MJD data for diaObjectId={diaObjectId}")
        plt.close(fig)
        return

    mjd_min = np.min(all_mjds) - 10
    mjd_max = np.max(all_mjds) + 10
    ax_main.set_xlim(mjd_min, mjd_max)

    # ── Hide ax_main y-axis ticks (only used as base frame) ───────────────────
    ax_main.yaxis.set_visible(False)
    ax_main.tick_params(axis='y', which='both', left=False, right=False)

    # ── Create one twin y-axis per band ───────────────────────────────────────
    spine_offset_px = 60   # pixels between successive y-axis spines
    ax_bands = {}
    band_has_data = {}

    # Sorted so that the first band (u) is outermost spine
    bands_present = [b for b in bands
                     if b in df_src_obj['band'].unique() or b in df_frc_obj['band'].unique()]

    for idx, band in enumerate(bands):
        ax_b = ax_main.twinx()
        ax_b.spines['right'].set_position(('outward', offsets[band] * spine_offset_px))
        ax_b.spines['right'].set_color(colors[band])
        ax_b.tick_params(axis='y', colors=colors[band], labelsize=10)
        ax_b.yaxis.label.set_color(colors[band])
        ax_b.set_ylabel(f"{band}  [{y_label_base}]", fontsize=10, rotation=90,
                        labelpad=4)
        ax_bands[band] = ax_b
        band_has_data[band] = False

    legend_handles = []

    # ── Plot DiaSources (marker 'o') ──────────────────────────────────────────
    for band in bands:
        df_b = df_src_obj[df_src_obj['band'] == band].copy()
        if len(df_b) == 0:
            continue

        mjd  = df_b['midpointMjdTai'].values
        flux = df_b['psfFlux'].values
        ferr = df_b['psfFluxErr'].values

        if plot_mode == 'mag':
            y    = flux_to_mag(flux)
            yerr = flux_err_to_mag_err(flux, ferr)
        else:
            y    = flux
            yerr = ferr

        valid = np.isfinite(y) & np.isfinite(yerr)
        if valid.sum() == 0:
            continue

        ax_b = ax_bands[band]
        ax_b.errorbar(
            mjd[valid], y[valid], yerr=yerr[valid],
            fmt='o', color=colors[band], alpha=0.8,
            markersize=5, linewidth=0, elinewidth=1.2,
            capsize=3, label=f"{band} DiaSource"
        )
        band_has_data[band] = True

    # ── Plot ForcedSrc (marker '+') ───────────────────────────────────────────
    for band in bands:
        df_b = df_frc_obj[df_frc_obj['band'] == band].copy()
        if len(df_b) == 0:
            continue

        mjd  = df_b['mjd_visit'].values
        flux = df_b['psfFlux'].values
        ferr = df_b['psfFluxErr'].values

        if plot_mode == 'mag':
            y    = flux_to_mag(flux)
            yerr = flux_err_to_mag_err(flux, ferr)
        else:
            y    = flux
            yerr = ferr

        valid = np.isfinite(y) & np.isfinite(yerr)
        if valid.sum() == 0:
            continue

        ax_b = ax_bands[band]
        ax_b.errorbar(
            mjd[valid], y[valid], yerr=yerr[valid],
            fmt='+', color=colors[band], alpha=0.6,
            markersize=8, linewidth=0, elinewidth=1.0,
            capsize=2, label=f"{band} Forced"
        )
        band_has_data[band] = True

    # ── Invert y-axis for magnitude mode ──────────────────────────────────────
    if plot_mode == 'mag':
        for band in bands:
            if band_has_data[band]:
                ax_bands[band].invert_yaxis()

    # ── Date axis on top ──────────────────────────────────────────────────────
    ax_date = make_date_axis(ax_main, mjd_min, mjd_max, n_ticks=7)

    # ── Legend ────────────────────────────────────────────────────────────────
    # Build a compact legend with one entry per band × source type
    legend_elems = []
    for band in bands:
        if band_has_data[band]:
            legend_elems.append(
                Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[band],
                       markersize=7, label=f"{band} DiaSource")
            )
            legend_elems.append(
                Line2D([0], [0], marker='+', color=colors[band],
                       markersize=9, linewidth=0, markeredgewidth=1.5,
                       label=f"{band} Forced")
            )

    ax_main.legend(
        handles=legend_elems,
        loc='upper left',
        bbox_to_anchor=(0.01, 0.99),
        ncol=3,
        framealpha=0.85,
        fontsize=10,
        title='Band  /  Source type',
        title_fontsize=11,
    )

    # ── Title & layout ────────────────────────────────────────────────────────
    mode_str = 'PSF Flux (nJy)' if plot_mode == 'flux' else 'AB Magnitude'
    fig.suptitle(
        f"Light curve — diaObjectId={diaObjectId}\n"
        f"RA={ra:.5f}°  Dec={dec:.5f}°   nDiaSources={n}   [{mode_str}]",
        fontsize=14, y=1.02
    )

    # Add right margin to accommodate the rightmost spine
    plt.subplots_adjust(right=0.62, top=0.88)
    plt.tight_layout(rect=[0, 0, 0.62, 0.92])
    plt.show()

print("plot_lightcurve() helper defined.")

---
## 6. Main loop — plot light curves

In [ ]:
# Pre-group DiaSources and ForcedSrc by diaObjectId for fast lookup
src_grouped = df_src.groupby('diaObjectId')
frc_grouped = df_frc.groupby('diaObjectId')

print(f"Plotting {len(df_obj_plot)} light curves  "
      f"(mode={PLOT_MODE}, cut nDiaSources>{CUT_NDIASOURCES}) ...")

for idx, row in df_obj_plot.iterrows():
    oid = row['diaObjectId']

    # Retrieve data for this object
    try:
        df_src_obj = src_grouped.get_group(oid).copy()
    except KeyError:
        df_src_obj = pd.DataFrame(columns=df_src.columns)

    try:
        df_frc_obj = frc_grouped.get_group(oid).copy()
    except KeyError:
        df_frc_obj = pd.DataFrame(columns=df_frc.columns)

    n_src = len(df_src_obj)
    n_frc = len(df_frc_obj)
    print(f"\n[{idx+1}/{len(df_obj_plot)}] diaObjectId={oid}  "
          f"nDiaSources={int(row['nDiaSources'])}  "
          f"DiaSrc={n_src}  ForcedSrc={n_frc}")

    plot_lightcurve(
        diaobj_row=row,
        df_src_obj=df_src_obj,
        df_frc_obj=df_frc_obj,
        plot_mode=PLOT_MODE,
    )

---
## 7. Optional — Switch to AB magnitude mode

In [ ]:
# Rerun the loop with PLOT_MODE='mag' to see magnitudes
PLOT_MODE = 'mag'

print(f"Re-plotting {len(df_obj_plot)} light curves in AB magnitude mode ...")

for idx, row in df_obj_plot.iterrows():
    oid = row['diaObjectId']

    try:
        df_src_obj = src_grouped.get_group(oid).copy()
    except KeyError:
        df_src_obj = pd.DataFrame(columns=df_src.columns)

    try:
        df_frc_obj = frc_grouped.get_group(oid).copy()
    except KeyError:
        df_frc_obj = pd.DataFrame(columns=df_frc.columns)

    print(f"\n[{idx+1}/{len(df_obj_plot)}] diaObjectId={oid}  "
          f"nDiaSources={int(row['nDiaSources'])}")

    plot_lightcurve(
        diaobj_row=row,
        df_src_obj=df_src_obj,
        df_frc_obj=df_frc_obj,
        plot_mode='mag',
    )

---
## 8. Optional — Quick nDiaSources distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_obj['nDiaSources'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(CUT_NDIASOURCES, color='red', linewidth=2, linestyle='--',
           label=f'Cut = {CUT_NDIASOURCES}')
ax.set_xlabel('nDiaSources', fontsize=13)
ax.set_ylabel('N DiaObjects', fontsize=13)
ax.set_title('Distribution of nDiaSources (DiaObjects, COSMOS, nmin=200)', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
*End of notebook `2026-03-14_DP2_DDF_DiaObjects_LightCurves.ipynb`*